In [1]:
# from google.colab import drive
# drive.mount('/content/drive/')

# %cd /content/drive/MyDrive/Colab Notebooks/Kaggle Tubular Playground Series/S4E8 Binary Prediction of Poisonous Mushrooms/

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.compose import ColumnTransformer 
import eda_util
import optuna
import time

# Import PyTorch and related libraries
import torch
from torch import nn
from torch import optim
import torch.utils.data as data_utils
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {DEVICE} device")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using cuda device


In [3]:
RANDOM_STATE = 1048576
TARGET = 'accident_risk'

# Data Loading

In [4]:
train_csv = pd.read_csv('datasets/train_split.csv', index_col=0)
val_csv = pd.read_csv('datasets/val_split.csv', index_col=0)
test_csv = pd.read_csv('datasets/test_split.csv', index_col=0)

In [5]:
display(train_csv.head())
display(val_csv.head())
display(test_csv.head())

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
313352,rural,2,0.22,45,night,foggy,False,True,evening,True,False,2,0.36
348526,rural,4,0.22,35,night,foggy,True,False,afternoon,True,False,2,0.42
419885,urban,1,0.18,60,dim,rainy,True,False,evening,False,False,0,0.40
37241,rural,2,0.92,25,dim,clear,True,True,afternoon,False,True,1,0.35
85356,urban,4,0.14,25,night,rainy,True,True,evening,True,False,0,0.34


,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
262368,rural,4,0.37,35,daylight,clear,True,False,evening,False,False,1,0.06
341957,rural,4,0.75,60,daylight,clear,False,False,morning,True,True,2,0.48
66336,rural,2,0.66,35,daylight,foggy,True,True,morning,False,True,1,0.32
325259,highway,2,0.90,45,night,clear,True,True,evening,True,True,0,0.45
477777,rural,2,0.28,25,night,rainy,True,True,evening,False,False,2,0.46


,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
id,,,,,,,,,,,,,
73059,highway,4,0.49,70,dim,rainy,False,True,afternoon,False,True,0,0.49
292825,urban,3,0.56,45,dim,foggy,True,True,evening,True,True,1,0.21
282017,highway,1,0.90,25,daylight,clear,False,False,morning,False,True,0,0.23
69101,rural,2,0.58,70,night,clear,False,True,afternoon,True,False,1,0.52
505439,rural,4,0.54,45,daylight,foggy,True,False,afternoon,True,True,2,0.24


In [6]:
print(f'Training set shape: {train_csv.shape}, Validation set shape: {val_csv.shape}, Test set shape: {test_csv.shape}')
eda_util.DataImport.datasplit_info(train_csv, val_csv, test_csv)

Training set shape: (414203, 13), Validation set shape: (51775, 13), Test set shape: (51776, 13)


,Column,Train Null Count,Val Null Count,Test Null Count,Dtype
0,road_type,0,0,0,object
1,num_lanes,0,0,0,int64
2,curvature,0,0,0,float64
3,speed_limit,0,0,0,int64
4,lighting,0,0,0,object
5,weather,0,0,0,object
6,road_signs_present,0,0,0,bool
7,public_road,0,0,0,bool
8,time_of_day,0,0,0,object
9,holiday,0,0,0,bool


# Feature-Target Split

In [7]:
X_train, y_train = train_csv.drop(columns=[TARGET]), train_csv[TARGET]
X_val, y_val = val_csv.drop(columns=[TARGET]), val_csv[TARGET]
X_test, y_test = test_csv.drop(columns=[TARGET]), test_csv[TARGET]
print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}')

X_train shape: (414203, 12), X_val shape: (51775, 12), X_test shape: (51776, 12)


# Preprocessing

In [8]:
cat_features = ['road_type', 'lighting' , 'weather', 'time_of_day']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']
num_features = X_train.columns.difference(cat_features + bool_features).tolist()

## Categorical Encoding

In [9]:
ord_features = ['lighting', 'time_of_day']
nom_features = ['road_type', 'weather']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']
num_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']

In [10]:
encoders = ColumnTransformer([
    ('nom_features', OneHotEncoder(dtype=float, sparse_output=False), nom_features),
    ('ord_features', OneHotEncoder(dtype=float, sparse_output=False), ord_features),
    ('bool_features', OneHotEncoder(dtype=float, sparse_output=False), bool_features)
], 
remainder='passthrough',
verbose_feature_names_out=False)
encoders

,transformers,"[('nom_features', ...), ('ord_features', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [11]:
encoders.set_output(transform='pandas')
encoders.fit_transform(X_train).head()

,road_type_highway,road_type_rural,road_type_urban,weather_clear,weather_foggy,weather_rainy,lighting_daylight,lighting_dim,lighting_night,time_of_day_afternoon,...,public_road_False,public_road_True,holiday_False,holiday_True,school_season_False,school_season_True,num_lanes,curvature,speed_limit,num_reported_accidents
id,,,,,,,,,,,,,,,,,,,,,
313352,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,2,0.22,45,2
348526,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,4,0.22,35,2
419885,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1,0.18,60,0
37241,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,2,0.92,25,1
85356,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,4,0.14,25,0


## Feature Engineering

In [12]:
def add_related_features(X):
    X = X.copy()
    X['curvature_speed_limit'] = X['curvature'] * X['speed_limit']
    X['curvature_speed_limit_2'] = X['speed_limit'] ** 2 * X['curvature']
    return X

add_transformer = FunctionTransformer(add_related_features)
add_transformer

,func,<function add...001B26E8BBC40>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None
,inv_kw_args,None


## Full preprocessing pipeline

In [13]:
full_preprocessor = Pipeline(steps=[
    ('encoder', encoders),
    ('add_rel_feature', add_transformer),
    ('scaling', StandardScaler())
])
full_preprocessor

,steps,"[('encoder', ...), ('add_rel_feature', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nom_features', ...), ('ord_features', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


# Model Building

## Preprocessing

In [14]:
X_train_preprocessed = full_preprocessor.fit_transform(X_train)
X_train_preprocessed[:5]

array([[-0.70994854,  1.41474124, -0.70453225, -0.72820475,  1.36240195,
        -0.65988152, -0.72414085, -0.74161493,  1.52328514, -0.70395685,
         1.4121087 , -0.70920634,  0.99855489, -0.99855489, -0.99515662,
         0.99515662, -0.99261304,  0.99261304,  0.99511818, -0.99511818,
        -0.43823197, -0.98669699, -0.07047256,  0.90658735, -0.82512529,
        -0.67633396],
       [-0.70994854,  1.41474124, -0.70453225, -0.72820475,  1.36240195,
        -0.65988152, -0.72414085, -0.74161493,  1.52328514,  1.42054161,
        -0.70816078, -0.70920634, -1.0014472 ,  1.0014472 ,  1.00486695,
        -1.00486695, -0.99261304,  0.99261304,  0.99511818, -0.99511818,
         1.34703478, -0.98669699, -0.70425127,  0.90658735, -0.96829943,
        -0.84189361],
       [-0.70994854, -0.70684304,  1.41938145, -0.72820475, -0.73399778,
         1.51542355, -0.72414085,  1.34840867, -0.65647591, -0.70395685,
         1.4121087 , -0.70920634, -1.0014472 ,  1.0014472 ,  1.00486695,
       

In [15]:
def create_data_loader(X: pd.DataFrame | np.ndarray, y: pd.Series | np.ndarray,
                       batch_size: int = 32, num_workers: int = 0, persistent_workers: bool=False, shuffle: bool = True):
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    data_tensor = data_utils.TensorDataset(X, y)
    data_loader = data_utils.DataLoader(dataset=data_tensor,
                                        batch_size=batch_size,
                                        num_workers=num_workers,
                                        shuffle=shuffle,
                                        persistent_workers=persistent_workers)
    return data_loader

train_loader = create_data_loader(X_train_preprocessed, y_train, batch_size=2048, num_workers=6, persistent_workers=True)
val_loader = create_data_loader(full_preprocessor.transform(X_val), y_val, batch_size=2048, num_workers=6, persistent_workers=True)
test_loader = create_data_loader(full_preprocessor.transform(X_test), y_test, batch_size=2048, num_workers=6, persistent_workers=True)

## Base model

In [16]:
# Define the neural network model
class RegressionNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int]=[128, 64, 32], dropout: float=0.2):
        """
        Fully Connected Neural Network for Regression tasks.

        Args:
            input_dim (int): Number of input features after preprocessing/encoding.
            hidden_dims (list of int): Hidden layer sizes.
            dropout (float): Dropout probability for regularization.
        """
        super(RegressionNN, self).__init__()

        # Define the layers
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))

        self.model = nn.Sequential(*layers)

    # def init_weights(self):
    #     # Initialize weights with Xavier (Glorot) initialization
    #     nn.init.xavier_uniform_(self.fc1.weight)
    #     nn.init.xavier_uniform_(self.fc2.weight)
    #     nn.init.xavier_uniform_(self.fc3.weight)

    #     # Optionally, initialize biases to zero
    #     nn.init.zeros_(self.fc1.bias)
    #     nn.init.zeros_(self.fc2.bias)
    #     nn.init.zeros_(self.fc3.bias)


    def forward(self, X: pd.DataFrame | np.ndarray) -> pd.DataFrame | np.ndarray:
        '''
        Forward pass through the network.
        '''
        return self.model(X).squeeze(1)



In [17]:
input_dim = X_train_preprocessed.shape[1]
model = RegressionNN(input_dim=input_dim, hidden_dims=[128, 64, 32], dropout=0.1)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # 'min' for loss, 'max' for accuracy
    factor=0.5,        # multiply lr by this factor when plateauing
    patience=3,        # number of epochs with no improvement before reducing
    threshold=1e-4     # minimum change to qualify as improvement
)

model.to(DEVICE)

RegressionNN(
  (model): Sequential(
    (0): Linear(in_features=26, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.1, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [18]:
def train_loop(model: RegressionNN, 
               train_loader: data_utils.DataLoader, 
               val_loader: data_utils.DataLoader,
               criterion: nn.Module, 
               optimizer: optim.Optimizer, 
               scheduler: optim.lr_scheduler._LRScheduler,
               num_epochs: int = 100, 
               patience: int = 10,
               optuna_trial: optuna.trial.Trial | None = None,
               verbose: int = 1) -> tuple[float, float]:
    """
    Train the neural network model with early stopping.

    Args:
        model (RegressionNN): The neural network model to train.
        train_loader (DataLoader): DataLoader for training data.
        val_loader (DataLoader): DataLoader for validation data.
        criterion (nn.Module): Loss function.
        optimizer (optim.Optimizer): Optimizer for training.
        scheduler (optim.lr_scheduler._LRScheduler): Learning rate scheduler.
        num_epochs (int): Maximum number of epochs to train.
        patience_limit (int): Number of epochs with no improvement to wait before stopping.

    Returns:
        tuple: Best validation loss and corresponding RMSE score.
    """
    epoch_durations = []
    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        start_time = time.time()

        model.train()
        train_loss = 0.0
        train_score = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)
            train_score += rmse(y_batch.cpu().detach().numpy(), outputs.cpu().detach().numpy()) * X_batch.size(0)

        train_loss /= len(train_loader.dataset)
        train_score /= len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_score = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_score += rmse(y_batch.cpu(), outputs.cpu()) * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_score /= len(val_loader.dataset)

        end_time = time.time()
        epoch_duration = end_time - start_time
        epoch_durations.append(epoch_duration)
        mins, secs = divmod(epoch_duration, 60)

        scheduler.step(val_loss)

        if optuna_trial:
            optuna_trial.report(val_loss, epoch)
            if optuna_trial.should_prune():
                if verbose > 1:
                    print(f"Trial pruned at epoch {epoch+1}")
                raise optuna.TrialPruned()

        if verbose > 1:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Score: {train_score:.4f}, Val Loss: {val_loss:.4f}, Val Score: {val_score:.4f}, Time: {int(mins)}m {secs:.1f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "models/best_nn_model.pt")
            patience = 0
        else:
            patience += 1
            if patience >= 10:
                if verbose > 0:
                    print("Early stopping triggered")
                break
    if verbose > 0:
        print(f'Average epoch duration: {np.mean(epoch_durations):.2f} seconds, Total training time: {np.sum(epoch_durations)//60} minutes {np.sum(epoch_durations) % 60:.2f} seconds, Best Val Score: {np.sqrt(best_val_loss):.4f}')
    return best_val_loss, np.sqrt(best_val_loss)

In [19]:
# best_val_loss, best_val_score = train_loop(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=10, verbose=2)
# print(f'Best Val Loss: {best_val_loss:.4f}, Best Val Score: {best_val_score:.4f}')

In [20]:
# print(f'Test score: {rmse(y_test, model(torch.tensor(full_preprocessor.transform(X_test), dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()):.4f}')

## Optuna optimization

In [ ]:
def objective(trial):
    input_dim = 26  # X_train_preprocessed.shape[1]
    n_layers = trial.suggest_int("n_layers", 2, 4)
    hidden_dims = []
    for i in range(n_layers):
        hidden_dims.append(trial.suggest_categorical(f"n_units_l{i}", (32, 64, 128, 256)))
    model_params = {
        'hidden_dims': hidden_dims,
        'dropout': trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    }
    optimizer_params = {
        'lr': trial.suggest_float('learning_rate', 1e-6, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    }

    model = RegressionNN(input_dim=input_dim, **model_params)
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',        # 'min' for loss, 'max' for accuracy
        factor=0.5,        # multiply lr by this factor when plateauing
        patience=3,        # number of epochs with no improvement before reducing
        threshold=1e-4     # minimum change to qualify as improvement
    )
    
    scores = []

    train_loader = create_data_loader(X_train_preprocessed, y_train, batch_size=2048)
    val_loader = create_data_loader(full_preprocessor.transform(X_val), y_val, batch_size=2048)
    

    best_val_loss, best_val_score = train_loop(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=10, optuna_trial=trial, verbose=1)

    scores.append(best_val_score)

    return np.mean(scores)

# --- Create a shared storage for multi-process parallelization ---
storage_name = "sqlite:///database/optuna_nn.db"

# Create study object and optimize the objective function
study = optuna.create_study(
    direction="minimize", 
    study_name="Neural Network Optimization", 
    storage=storage_name, 
    load_if_exists=True
    )

# Optimize the objective function over 20 trials
study.optimize(objective, n_trials=20, n_jobs=4, show_progress_bar=True)

# Print the best hyperparameters
print("Best Parameters:", study.best_params)
print("Best RMSE:", study.best_value)

[I 2025-10-15 15:09:10,874] A new study created in RDB with name: Neural Network Optimization
  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# --- Train final model using best parameters ---
best_nn = RegressionNN(input_dim=26, **study.best_params, random_state=RANDOM_STATE)

train_csv = pd.read_csv('datasets/train_split.csv', index_col=0)
val_csv = pd.read_csv('datasets/val_split.csv', index_col=0)
test_csv = pd.read_csv('datasets/test_split.csv', index_col=0)

X_train, y_train = train_csv.drop(columns=[TARGET]), train_csv[TARGET]
X_val, y_val = val_csv.drop(columns=[TARGET]), val_csv[TARGET]
X_test, y_test = test_csv.drop(columns=[TARGET]), test_csv[TARGET]
print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}')

X_train_preprocessed = full_preprocessor.fit_transform(X_train)
train_loader = create_data_loader(X_train_preprocessed, y_train, batch_size=2048, num_workers=6, persistent_workers=True)
val_loader = create_data_loader(full_preprocessor.transform(X_val), y_val, batch_size=2048, num_workers=6, persistent_workers=True)
test_loader = create_data_loader(full_preprocessor.transform(X_test), y_test, batch_size=2048, num_workers=6, persistent_workers=True)

best_val_loss, best_val_score = train_loop(best_nn, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=100, patience=10, verbose=2)

print(f'Best Val Loss: {best_val_loss:.4f}, Best Val Score: {best_val_score:.4f}')
print(f'Test score: {rmse(y_test, best_nn(torch.tensor(full_preprocessor.transform(X_test), dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()):.4f}')

In [ ]:
import joblib
import json

with open('models/nn_optuna_best_params.json', 'w') as f:
    json.dump(study.best_params, f, indent=4)
joblib.dump(best_nn, 'best_nn_optuna_model.pkl')

# with open('models/nn_optuna_best_params.json') as param_file:
#    best_nn_optuna_param = json.load(param_file)
# best_reg = joblib.load('best_nn_model.pkl')

# Submit Prediction

In [ ]:
full_test_csv = pd.read_csv('datasets/test.csv', index_col=0)
full_test_encoded = full_preprocessor.transform(full_test_csv)
full_test_encoded[:5]

## Base Model Prediction

In [ ]:
y_pred = model(torch.tensor(full_test_encoded, dtype=torch.float32).to(DEVICE)).cpu().detach().numpy()

submission = pd.read_csv('datasets/sample_submission.csv', index_col=0)
print(submission.head())
submission[TARGET] = y_pred.round(3)
print('\n')
print(submission.head())
submission.to_csv('datasets/submission_NN_Base.csv')
print('Result saved successfully!')

## Best LGB Classifier

In [ ]:
y_pred = best_reg.predict(full_test_encoded)

In [ ]:
submission = pd.read_csv('datasets/sample_submission.csv', index_col=0)
print(submission.head())
submission[TARGET] = y_pred.round(3)
print('\n')
print(submission.head())
submission.to_csv('datasets/submission_LGBM_Optuna.csv')
print('Result saved successfully!')

Public Score: 0.05565